# Matched unsupervised cross-dataset significance experiment

This is a **new experiment** using local raw CICIDS2017, CSE-CIC-IDS2018 and UNSW-NB15 files. It does not reproduce scores from the earlier full-data notebook.

- Exactly nine harmonized flow features are used in every dataset.
- The loader samples up to 1,000 rows per 100,000-row CSV chunk and retains at most 25,000 deduplicated rows per dataset. The file manifest and sampled data are saved for audit.
- Ten stratified folds are shared by Isolation Forest, One-Class SVM and the Autoencoder. For each fold and source, fitting uses only source normal training traffic; the source validation set calibrates the anomaly threshold. This is **unsupervised model fitting with label-informed calibration**, not completely label-free evaluation.
- Every trained source model is evaluated on all three datasets. The test fold is untouched until evaluation. Attack-vs-benign metrics use attack labels only during evaluation.
- Paired fold sign-flip tests compare models (H1), within versus cross setting (H2), attack types (H3), model-by-setting effects (H4), and setting-by-attack effects (H6). All reported p-values are Holm-adjusted across the executed contrasts. H5 is descriptive; H7 and H8 require a separate feature-intervention design.

These are conditional comparisons on repeated folds of the **same benchmark captures**. They do not establish performance on an independently collected network. Flow-level random folds may retain capture-specific dependencies; report this limitation in the thesis.


## Observed results from the ten-fold run

The table reports the **smallest Holm-adjusted contrast p-value** for each testable hypothesis, not an omnibus p-value. No individual contrast passed 0.05 after adjusting across all 180 executed contrasts. This is **inconclusive**, not proof that model, setting, or attack type has no effect. With ten paired folds, the smallest attainable two-sided exact p-value is 0.001953; correcting 180 contrasts makes this design underpowered for confirmatory claims. A future study should predeclare a small set of primary contrasts and use independent capture groups or more genuinely independent replications.

|   S.No. | Hypothesis   | Significance test (p-value)        | Remarks                                                                 |
|--------:|:-------------|:-----------------------------------|:------------------------------------------------------------------------|
|       1 | H1           | min Holm p = 0.35156               | 0/12 contrasts significant; inspect individual effects.                 |
|       2 | H2           | min Holm p = 0.35156               | 0/4 contrasts significant; inspect individual effects.                  |
|       3 | H3           | min Holm p = 0.35156               | 0/76 contrasts significant; inspect individual effects.                 |
|       4 | H4           | min Holm p = 0.35156               | 0/12 contrasts significant; inspect individual effects.                 |
|       5 | H5           | Not a p-value hypothesis as worded | Report mean, worst-path and variability across all cross-dataset paths. |
|       6 | H6           | min Holm p = 0.35156               | 0/76 contrasts significant; inspect individual effects.                 |
|       7 | H7           | Not estimable                      | Requires a predeclared feature intervention and mediation design.       |
|       8 | H8           | Not estimable                      | Requires a predeclared feature intervention and mediation design.       |

### Cross-path robustness (descriptive H5)

| Model                |   mean_cross_auc |   worst_cross_auc |   sd_cross_auc |   mean_cross_f1 |
|:---------------------|-----------------:|------------------:|---------------:|----------------:|
| Improved Autoencoder |           0.647  |            0.2644 |         0.2012 |          0.2537 |
| One-Class SVM        |           0.6074 |            0.2708 |         0.2173 |          0.178  |
| Isolation Forest     |           0.5975 |            0.2503 |         0.2251 |          0.2294 |

All scores here belong to the **new capped experiment**; do not mix them with the original full-data notebook. Full raw and adjusted contrast p-values are in `unsupervised_matched_results/hypothesis_contrasts_holm.csv`.


In [ ]:
"""Matched, repeated 3x3 unsupervised IDS experiment on local benchmark files.

This is a new, capped experiment. It does not reproduce the original full-data
notebook scores. Run as a script or copy into a notebook cell.
"""

from __future__ import annotations

import json
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.ensemble import IsolationForest
from sklearn.metrics import f1_score, precision_recall_curve, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.svm import OneClassSVM
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


DATA_ROOT = Path("/Users/ghost/Documents/Personal/69/Kuek/Academic/Gisma/Units/Year 2/Research Methods/Datasets")
OUTPUT_ROOT = (Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()) / "unsupervised_matched_results"
DATASETS = ("CICIDS2017", "CSE-CIC-IDS2018", "UNSW-NB15")
FEATURES = (
    "duration_seconds", "forward_packets", "backward_packets", "forward_bytes",
    "backward_bytes", "forward_packet_length_mean", "backward_packet_length_mean",
    "flow_bytes_per_second", "flow_packets_per_second",
)
CHUNK_ROWS = 100_000
PER_CHUNK_SAMPLE = 1_000
MAX_DATASET_ROWS = 25_000
FOLDS = 10
RANDOM_STATE = 2026

UNSW_COLS = [
    "srcip", "sport", "dstip", "dsport", "proto", "state", "dur", "sbytes",
    "dbytes", "sttl", "dttl", "sloss", "dloss", "service", "sload", "dload",
    "spkts", "dpkts", "swin", "dwin", "stcpb", "dtcpb", "smeansz", "dmeansz",
    "trans_depth", "res_bdy_len", "sjit", "djit", "stime", "ltime", "sintpkt",
    "dintpkt", "tcprtt", "synack", "ackdat", "is_sm_ips_ports", "ct_state_ttl",
    "ct_flw_http_mthd", "is_ftp_login", "ct_ftp_cmd", "ct_srv_src", "ct_srv_dst",
    "ct_dst_ltm", "ct_src_ltm", "ct_src_dport_ltm", "ct_dst_sport_ltm",
    "ct_dst_src_ltm", "attack_cat", "label",
]


def normalise(name):
    text = str(name).strip().lower()
    for old in (" ", "/", "-", "."):
        text = text.replace(old, "_")
    while "__" in text:
        text = text.replace("__", "_")
    return text


def select_columns(path, dataset):
    names = pd.read_csv(path, nrows=0, encoding="latin-1").columns.tolist()
    lookup = {normalise(name): name for name in names}
    aliases = {
        "duration": ("flow_duration",),
        "forward_packets": ("total_fwd_packets", "total_fwd_pkts", "tot_fwd_pkts"),
        "backward_packets": ("total_backward_packets", "total_bwd_packets", "tot_bwd_pkts"),
        "forward_bytes": ("total_length_of_fwd_packets", "total_len_of_fwd_packets", "totlen_fwd_pkts"),
        "backward_bytes": ("total_length_of_bwd_packets", "total_len_of_bwd_packets", "totlen_bwd_pkts"),
        "attack_type": ("label",),
    }
    chosen = {}
    for role, alternatives in aliases.items():
        match = next((lookup[key] for key in alternatives if key in lookup), None)
        if match is None:
            raise ValueError(f"{dataset}: missing {role} in {path.name}")
        chosen[role] = match
    return chosen


def harmonise_sample(chunk, chosen, dataset, source_file):
    if dataset == "UNSW-NB15":
        mapping = {"duration": "dur", "forward_packets": "spkts", "backward_packets": "dpkts",
                   "forward_bytes": "sbytes", "backward_bytes": "dbytes", "attack_type": "attack_cat"}
        chosen = mapping
        labels = pd.to_numeric(chunk["label"], errors="coerce")
        valid = labels.isin((0, 1))
        attack_type = chunk[chosen["attack_type"]].fillna("Unknown").astype(str).str.strip()
        y = labels.astype("float32")
    else:
        attack_type = chunk[chosen["attack_type"]].astype(str).str.strip()
        lower = attack_type.str.lower()
        valid = ~lower.isin(("", "nan", "none", "null", "label"))
        y = (~lower.isin(("benign", "normal", "normal.", "0"))).astype("float32")
    duration = pd.to_numeric(chunk[chosen["duration"]], errors="coerce")
    if dataset != "UNSW-NB15":
        duration = duration / 1_000_000.0
    fp = pd.to_numeric(chunk[chosen["forward_packets"]], errors="coerce")
    bp = pd.to_numeric(chunk[chosen["backward_packets"]], errors="coerce")
    fb = pd.to_numeric(chunk[chosen["forward_bytes"]], errors="coerce")
    bb = pd.to_numeric(chunk[chosen["backward_bytes"]], errors="coerce")

    def divide(top, bottom):
        return top / bottom.replace(0, np.nan)

    out = pd.DataFrame({
        "duration_seconds": duration,
        "forward_packets": fp,
        "backward_packets": bp,
        "forward_bytes": fb,
        "backward_bytes": bb,
        "forward_packet_length_mean": divide(fb, fp),
        "backward_packet_length_mean": divide(bb, bp),
        "flow_bytes_per_second": divide(fb + bb, duration),
        "flow_packets_per_second": divide(fp + bp, duration),
        "binary_label": y,
        "attack_type": attack_type,
        "source_file": source_file,
        "row_in_file": chunk.index,
    })
    out = out.loc[valid].reset_index(drop=True)
    out["binary_label"] = out["binary_label"].astype(np.int8)
    for column in FEATURES:
        out[column] = pd.to_numeric(out[column], errors="coerce").astype("float32")
    return out


def sample_dataset(dataset):
    folder = DATA_ROOT / dataset
    if dataset == "UNSW-NB15":
        files = [folder / f"UNSW-NB15_{i}.csv" for i in range(1, 5)]
    else:
        files = sorted(folder.glob("*.csv"))
    if not files or any(not path.is_file() for path in files):
        raise FileNotFoundError(f"Missing files for {dataset}: {files}")
    samples, manifest = [], []
    for file_number, path in enumerate(files):
        if dataset == "UNSW-NB15":
            kwargs = {"names": UNSW_COLS, "header": None,
                      "usecols": [6, 7, 8, 16, 17, 47, 48]}
            chosen = None
        else:
            chosen = select_columns(path, dataset)
            kwargs = {"usecols": list(chosen.values())}
        read_rows = 0
        sampled_rows = 0
        reader = pd.read_csv(path, chunksize=CHUNK_ROWS, low_memory=False,
                             encoding="latin-1", **kwargs)
        for chunk_number, chunk in enumerate(reader):
            read_rows += len(chunk)
            n = min(PER_CHUNK_SAMPLE, len(chunk))
            sampled = chunk.sample(n=n, random_state=RANDOM_STATE + file_number * 100_003 + chunk_number)
            sampled_rows += n
            samples.append(harmonise_sample(sampled, chosen, dataset, path.name))
        manifest.append({"Dataset": dataset, "File": str(path), "Rows read": read_rows,
                         "Rows sampled": sampled_rows, "Bytes": path.stat().st_size})
        print(f"{dataset}: {path.name}: {read_rows:,} read, {sampled_rows:,} sampled", flush=True)
    frame = pd.concat(samples, ignore_index=True)
    frame = frame.replace([np.inf, -np.inf], np.nan)
    frame = frame.drop_duplicates(subset=[*FEATURES, "binary_label"])
    if len(frame) > MAX_DATASET_ROWS:
        ids, _ = train_test_split(np.arange(len(frame)), train_size=MAX_DATASET_ROWS,
                                  stratify=frame["binary_label"], random_state=RANDOM_STATE)
        frame = frame.iloc[np.sort(ids)]
    frame = frame.reset_index(drop=True)
    if set(frame["binary_label"].unique()) != {0, 1}:
        raise ValueError(f"{dataset} sample must contain benign and attack rows")
    print(f"{dataset} retained: {len(frame):,}, attacks: {int(frame.binary_label.sum()):,}", flush=True)
    return frame, manifest


class SourcePreprocessor:
    def fit(self, frame):
        numeric = frame[list(FEATURES)].replace([np.inf, -np.inf], np.nan).astype("float64")
        self.medians = numeric.median().fillna(0)
        clean = numeric.fillna(self.medians).clip(lower=0)
        self.upper = clean.quantile(.999).replace(0, 1).fillna(1)
        self.scaler = RobustScaler(quantile_range=(10, 90)).fit(np.log1p(clean.clip(upper=self.upper, axis=1)))
        return self

    def transform(self, frame):
        numeric = frame[list(FEATURES)].replace([np.inf, -np.inf], np.nan).astype("float64")
        clean = numeric.fillna(self.medians).fillna(0).clip(lower=0)
        values = self.scaler.transform(np.log1p(clean.clip(upper=self.upper, axis=1)))
        return np.nan_to_num(values, nan=0, posinf=20, neginf=-20).astype("float32")


class Autoencoder(nn.Module):
    def __init__(self, dimension):
        super().__init__()
        self.network = nn.Sequential(nn.Linear(dimension, 32), nn.LeakyReLU(.1),
                                     nn.Linear(32, 8), nn.LeakyReLU(.1),
                                     nn.Linear(8, 32), nn.LeakyReLU(.1),
                                     nn.Linear(32, dimension))

    def forward(self, x):
        return self.network(x)


def train_autoencoder(x_train, x_validation, seed):
    torch.manual_seed(seed)
    model = Autoencoder(x_train.shape[1])
    optimizer = torch.optim.AdamW(model.parameters(), lr=.001, weight_decay=1e-5)
    train_loader = DataLoader(TensorDataset(torch.from_numpy(x_train)), batch_size=512, shuffle=True)
    validation_tensor = torch.from_numpy(x_validation)
    best_loss, patience, best_state = np.inf, 0, None
    for _ in range(20):
        model.train()
        for (batch,) in train_loader:
            optimizer.zero_grad()
            loss = nn.functional.smooth_l1_loss(model(batch), batch)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = nn.functional.smooth_l1_loss(model(validation_tensor), validation_tensor).item()
        if val_loss < best_loss - 1e-6:
            best_loss, patience = val_loss, 0
            best_state = {key: value.detach().clone() for key, value in model.state_dict().items()}
        else:
            patience += 1
            if patience >= 4:
                break
    model.load_state_dict(best_state)
    return model


def ae_scores(model, values):
    model.eval()
    output = []
    with torch.no_grad():
        for start in range(0, len(values), 4096):
            batch = torch.from_numpy(values[start:start + 4096])
            error = ((model(batch) - batch) ** 2).mean(dim=1)
            output.append(error.numpy())
    return np.concatenate(output)


def f1_threshold(y, scores):
    precision, recall, thresholds = precision_recall_curve(y, scores)
    values = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    if not len(thresholds):
        raise ValueError("Validation data needs both classes")
    return float(thresholds[int(np.nanargmax(values))])


def record(seed, source, target, model, y, scores, threshold, attack_type=None):
    pred = (scores >= threshold).astype(np.int8)
    return {"Seed": seed, "Train Dataset": source, "Test Dataset": target,
            "Model": model, "Attack Type": attack_type,
            "AUC Definition": "attack-vs-benign" if attack_type is not None else "binary-all-attacks",
            "Precision": precision_score(y, pred, zero_division=0),
            "Recall": recall_score(y, pred, zero_division=0),
            "F1-Score": f1_score(y, pred, zero_division=0),
            "ROC-AUC": roc_auc_score(y, scores), "Test rows": len(y)}


def evaluate(seed, source, target, model, y, scores, threshold, types):
    rows = [record(seed, source, target, model, y, scores, threshold)]
    for attack_type in sorted(set(types[y == 1])):
        positives = (y == 1) & (types == attack_type)
        if positives.sum() < 10 or (y == 0).sum() < 10:
            continue
        mask = (y == 0) | positives
        rows.append(record(seed, source, target, model, y[mask], scores[mask], threshold, attack_type))
    return rows


def make_partitions(frames):
    folds = {}
    for name, frame in frames.items():
        y = frame["binary_label"].to_numpy()
        splitter = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=RANDOM_STATE)
        folds[name] = list(splitter.split(np.zeros(len(y)), y))
    return folds


def run_experiments(frames, max_folds=FOLDS):
    partitions = make_partitions(frames)
    rows = []
    OUTPUT_ROOT.mkdir(exist_ok=True)
    for fold in range(max_folds):
        for source in DATASETS:
            source_frame = frames[source]
            source_y = source_frame["binary_label"].to_numpy(dtype=np.int8)
            development, _ = partitions[source][fold]
            train_ids, validation_ids = train_test_split(
                development, test_size=.2, stratify=source_y[development], random_state=RANDOM_STATE + fold)
            normal_ids = train_ids[source_y[train_ids] == 0]
            if len(normal_ids) < 100:
                raise ValueError(f"Too few normal training rows for {source}")
            preprocessor = SourcePreprocessor().fit(source_frame.iloc[normal_ids])
            x_normal = preprocessor.transform(source_frame.iloc[normal_ids])
            x_val = preprocessor.transform(source_frame.iloc[validation_ids])
            y_val = source_y[validation_ids]
            x_ae_val = x_val[y_val == 0]
            forest = IsolationForest(n_estimators=100, max_samples=min(1024, len(x_normal)),
                                     random_state=RANDOM_STATE + fold, n_jobs=-1).fit(x_normal)
            rng = np.random.default_rng(RANDOM_STATE + fold)
            svm_ids = rng.choice(len(x_normal), min(2000, len(x_normal)), replace=False)
            svm = OneClassSVM(kernel="rbf", gamma="scale", nu=.05).fit(x_normal[svm_ids])
            ae = train_autoencoder(x_normal, x_ae_val, RANDOM_STATE + fold)
            score_fns = {
                "Isolation Forest": lambda x: -forest.decision_function(x),
                "One-Class SVM": lambda x: -svm.decision_function(x),
                "Improved Autoencoder": lambda x: ae_scores(ae, x),
            }
            thresholds = {name: f1_threshold(y_val, score_fn(x_val))
                          for name, score_fn in score_fns.items()}
            for target in DATASETS:
                _, test_ids = partitions[target][fold]
                target_frame = frames[target].iloc[test_ids]
                x_test = preprocessor.transform(target_frame)
                y_test = target_frame["binary_label"].to_numpy(dtype=np.int8)
                types = target_frame["attack_type"].astype(str).to_numpy()
                for name, score_fn in score_fns.items():
                    scores = score_fn(x_test)
                    rows.extend(evaluate(fold, source, target, name, y_test,
                                         scores, thresholds[name], types))
            print(f"fold {fold + 1}/{max_folds}, source {source} complete", flush=True)
        pd.DataFrame(rows).to_csv(OUTPUT_ROOT / "unsupervised_repeated_runs.csv", index=False)
    return pd.DataFrame(rows)


def main(max_folds=FOLDS, reuse_samples=False):
    OUTPUT_ROOT.mkdir(exist_ok=True)
    frames = {}
    manifest = []
    cached = all((OUTPUT_ROOT / f"sample_{name}.parquet").is_file() for name in DATASETS)
    if reuse_samples and cached:
        for name in DATASETS:
            frames[name] = pd.read_parquet(OUTPUT_ROOT / f"sample_{name}.parquet")
            print(f"Reused {name}: {len(frames[name]):,} rows", flush=True)
    else:
        for name in DATASETS:
            frames[name], entries = sample_dataset(name)
            manifest.extend(entries)
        pd.DataFrame(manifest).to_csv(OUTPUT_ROOT / "raw_file_manifest.csv", index=False)
        for name, frame in frames.items():
            frame.to_parquet(OUTPUT_ROOT / f"sample_{name}.parquet", index=False)
    metadata = {"data_root": str(DATA_ROOT), "features": FEATURES,
                "chunk_rows": CHUNK_ROWS, "per_chunk_sample": PER_CHUNK_SAMPLE,
                "max_dataset_rows": MAX_DATASET_ROWS, "folds": FOLDS,
                "random_state": RANDOM_STATE,
                "interpretation": "New sampled experiment; not the original full-data scores."}
    (OUTPUT_ROOT / "protocol.json").write_text(json.dumps(metadata, indent=2))
    return run_experiments(frames, max_folds=max_folds)




In [ ]:
# Find the packaged results whether Jupyter started in Code/ or the dissertation root.
for candidate in (Path.cwd() / "unsupervised_matched_results",
                  Path.cwd() / "Code" / "unsupervised_matched_results"):
    if (candidate / "unsupervised_repeated_runs.csv").is_file():
        OUTPUT_ROOT = candidate
        break
print("Results folder:", OUTPUT_ROOT)
print("Repeated-run CSV found:", (OUTPUT_ROOT / "unsupervised_repeated_runs.csv").is_file())


In [ ]:
# Set True only if you want to regenerate the sampled data and all ten folds.
# On this computer, the completed CSV can be analysed without rerunning the models.
RUN_FULL_EXPERIMENT = False
if RUN_FULL_EXPERIMENT:
    repeated_results = main(max_folds=FOLDS, reuse_samples=False)
    display(repeated_results.head())
else:
    print("Existing results can be analysed in the next cell without retraining.")


## Hypothesis tests and results

Run this cell after the ten folds complete. Missing p-values remain missing; they are never fabricated.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from itertools import combinations, product

PARADIGM = "unsupervised"
from itertools import combinations, product

METRICS = ["Precision", "Recall", "F1-Score", "ROC-AUC"]
MODELS = ["Isolation Forest", "One-Class SVM", "Improved Autoencoder"] if PARADIGM == "unsupervised" else ["Random Forest", "XGBoost", "Deep Neural Network"]
MIN_SEEDS = 10  # Predeclare; six is the minimum for an exact two-sided sign-flip p < .05.


def paired_signflip_p(differences, random_seed=2026, draws=20000):
    """Exact paired sign-flip for <=16 nonzero pairs, otherwise Monte Carlo."""
    x = np.asarray(differences, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) < MIN_SEEDS:
        raise ValueError(f"Need >= {MIN_SEEDS} finite paired seed differences; received {len(x)}.")
    observed = abs(x.mean())
    if len(x) <= 16:
        null = np.array([abs(np.mean(np.asarray(signs) * x)) for signs in product((-1, 1), repeat=len(x))])
        return float(np.mean(null >= observed - 1e-12))
    rng = np.random.default_rng(random_seed)
    signs = rng.choice((-1, 1), size=(draws, len(x)))
    null = np.abs((signs * x).mean(axis=1))
    return float((1 + np.count_nonzero(null >= observed - 1e-12)) / (draws + 1))


def holm_adjust(p_values):
    p = np.asarray(p_values, dtype=float)
    order = np.argsort(p)
    adjusted = np.empty(len(p), dtype=float)
    running = 0.0
    for rank, index in enumerate(order):
        running = max(running, min(1.0, (len(p) - rank) * p[index]))
        adjusted[index] = running
    return adjusted


def analyse_repeated_runs(frame):
    required = {"Seed", "Train Dataset", "Test Dataset", "Model", *METRICS}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"Repeated-run table is missing columns: {sorted(missing)}")
    df = frame.copy()
    df = df[df["Model"].isin(MODELS)].copy()
    df["Setting"] = np.where(df["Train Dataset"] == df["Test Dataset"], "Within", "Cross")
    for metric in METRICS:
        df[metric] = pd.to_numeric(df[metric], errors="raise")
        if not df[metric].between(0, 1).all():
            raise ValueError(f"{metric} must be in [0, 1].")
    keys = ["Seed", "Train Dataset", "Test Dataset", "Model"]
    if "Attack Type" in df.columns:
        keys.append("Attack Type")
    if df.duplicated(keys).any():
        raise ValueError("Duplicate experiment rows found; inspect Seed/path/model/attack keys.")
    if df["Seed"].nunique() < MIN_SEEDS:
        raise ValueError(f"Need >= {MIN_SEEDS} distinct, predeclared seeds; found {df['Seed'].nunique()}.")

    tests = []
    def compare(hypothesis, contrast, metric, table, left, right):
        pair = table[[left, right]].dropna()
        if len(pair) < MIN_SEEDS:
            return
        diffs = pair[left].to_numpy() - pair[right].to_numpy()
        try:
            p = paired_signflip_p(diffs)
        except ValueError:
            return
        tests.append({"Hypothesis": hypothesis, "Contrast": contrast, "Metric": metric,
                      "Paired seeds": len(pair), "Mean difference": float(np.mean(diffs)), "p raw": p})

    # H1: each seed contributes one average over the SAME evaluation paths for each model.
    base = df[df["Attack Type"].isna()] if "Attack Type" in df.columns else df
    for metric in METRICS:
        same_paths = base.pivot_table(index=["Seed", "Train Dataset", "Test Dataset"], columns="Model", values=metric)
        available = [m for m in MODELS if m in same_paths.columns]
        for left, right in combinations(available, 2):
            matched = same_paths[[left, right]].dropna().reset_index()
            by_seed = matched.groupby("Seed")[[left, right]].mean()
            compare("H1", f"{left} vs {right}", metric, by_seed, left, right)

    # H2: within vs mean of the two external targets, matched by seed, source and model.
    # H4: model differences in the within-minus-cross gap, paired by seed and source.
    if set(df["Setting"]) == {"Within", "Cross"}:
        for metric in METRICS:
            setting = base.groupby(["Seed", "Train Dataset", "Model", "Setting"], as_index=False)[metric].mean()
            pivot = setting.pivot_table(index=["Seed", "Train Dataset", "Model"], columns="Setting", values=metric).dropna()
            gaps = pivot.assign(Gap=pivot["Within"] - pivot["Cross"]).reset_index()
            h2 = gaps.groupby("Seed")[["Within", "Cross"]].mean()
            compare("H2", "Within vs cross (source/model-averaged)", metric, h2, "Within", "Cross")
            gap_pivot = gaps.pivot_table(index=["Seed", "Train Dataset"], columns="Model", values="Gap")
            available = [m for m in MODELS if m in gap_pivot.columns]
            for left, right in combinations(available, 2):
                by_seed = gap_pivot[[left, right]].dropna().reset_index().groupby("Seed")[[left, right]].mean()
                compare("H4", f"Setting gap: {left} vs {right}", metric, by_seed, left, right)

    # H3/H6 require per-attack-class rows. Compare classes within a target dataset;
    # otherwise target-dataset shifts are confounded with attack type.
    if "Attack Type" in df.columns:
        attack = df[df["Attack Type"].notna()].copy()
        for target, target_attack in attack.groupby("Test Dataset"):
            classes = sorted(target_attack["Attack Type"].astype(str).unique())
            for metric in METRICS:
                if metric == "ROC-AUC":
                    if "AUC Definition" not in target_attack.columns:
                        continue
                    definitions = target_attack["AUC Definition"].dropna().unique()
                    if len(definitions) != 1 or definitions[0] not in ("attack-vs-benign", "one-vs-rest"):
                        continue
                class_means = target_attack.groupby(["Seed", "Attack Type"], as_index=False)[metric].mean()
                by_seed = class_means.pivot(index="Seed", columns="Attack Type", values=metric)
                for left, right in combinations(classes, 2):
                    compare("H3", f"{target}: {left} vs {right}", metric, by_seed, left, right)
                if set(target_attack["Setting"]) == {"Within", "Cross"}:
                    setting_attack = target_attack.groupby(["Seed", "Attack Type", "Setting"], as_index=False)[metric].mean()
                    gaps = setting_attack.pivot_table(index=["Seed", "Attack Type"], columns="Setting", values=metric).dropna()
                    gaps = (gaps["Within"] - gaps["Cross"]).unstack("Attack Type")
                    for left, right in combinations(classes, 2):
                        compare("H6", f"{target} setting gap: {left} vs {right}", metric, gaps, left, right)

    result = pd.DataFrame(tests)
    if result.empty:
        return result
    result["p Holm"] = holm_adjust(result["p raw"])
    result["Significant at .05"] = result["p Holm"] < 0.05
    return result.sort_values(["Hypothesis", "Metric", "p Holm"]).reset_index(drop=True)

result_path = OUTPUT_ROOT / "unsupervised_repeated_runs.csv"
if not result_path.is_file():
    print(f"No repeated-run results at {result_path}. Run the experiment cell first.")
else:
    repeated_runs = pd.read_csv(result_path)
    print("Completed folds:", sorted(repeated_runs["Seed"].unique()))
    print("Aggregate result rows:", repeated_runs["Attack Type"].isna().sum())
    contrasts = analyse_repeated_runs(repeated_runs)
    display(contrasts.round(5))
    contrasts.to_csv(OUTPUT_ROOT / "hypothesis_contrasts_holm.csv", index=False)

    summary = []
    for number in range(1, 9):
        hypothesis = f"H{number}"
        subset = contrasts[contrasts["Hypothesis"] == hypothesis]
        if number == 5:
            p_text = "Not a p-value hypothesis as worded"
            remarks = "Report mean, worst-path and variability across all cross-dataset paths."
        elif number in (7, 8):
            p_text = "Not estimable"
            remarks = "Requires a predeclared feature intervention and mediation design."
        elif subset.empty:
            p_text = "Not estimable from completed contrasts"
            remarks = "Insufficient matched attack classes or repeated differences."
        else:
            p_text = f"min Holm p = {subset['p Holm'].min():.5g}"
            remarks = f"{int(subset['Significant at .05'].sum())}/{len(subset)} contrasts significant; inspect individual effects."
        summary.append({"S.No.": number, "Hypothesis": hypothesis,
                        "Significance test (p-value)": p_text, "Remarks": remarks})
    hypothesis_summary = pd.DataFrame(summary)
    display(hypothesis_summary)
    hypothesis_summary.to_csv(OUTPUT_ROOT / "hypothesis_summary.csv", index=False)

    aggregate = repeated_runs[repeated_runs["Attack Type"].isna()]
    cross = aggregate[aggregate["Train Dataset"] != aggregate["Test Dataset"]]
    robustness = cross.groupby("Model").agg(
        mean_cross_auc=("ROC-AUC", "mean"),
        worst_cross_auc=("ROC-AUC", "min"),
        sd_cross_auc=("ROC-AUC", "std"),
        mean_cross_f1=("F1-Score", "mean"),
    ).sort_values("mean_cross_auc", ascending=False)
    print("H5 descriptive robustness:")
    display(robustness.round(4))
    robustness.to_csv(OUTPUT_ROOT / "cross_path_robustness.csv")
